# Road Segmentation Training on Google Colab

This notebook trains a DeepLabV3 model for road segmentation using the DeepGlobe dataset.

**Checkpoint Feature:** If training is interrupted, re-run all cells to resume from the last saved epoch.


## Step 1: Install Dependencies


In [ ]:
!pip install kagglehub torch torchvision tqdm -q


## Step 2: Download Dataset


In [ ]:
import kagglehub
import os

# Download DeepGlobe dataset
path = kagglehub.dataset_download("balraj98/deepglobe-road-extraction-dataset")
print(f"Dataset downloaded to: {path}")

# Find the train directory
train_dir = os.path.join(path, "train")
if not os.path.exists(train_dir):
    # Check if files are directly in the path
    train_dir = path
    
print(f"Training data directory: {train_dir}")
print(f"Files found: {len(os.listdir(train_dir))}")


## Step 3: Local checkpoint folder (no Drive mount)


In [ ]:
# Create a local folder in the Colab runtime for checkpoints/models
import os
os.makedirs("/content/checkpoints", exist_ok=True)

print("✅ Local checkpoint folder ready at /content/checkpoints")


## Step 4: Check GPU


In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ No GPU detected! Go to Runtime → Change runtime type → GPU")


## Step 4: Create Dataset Class


In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

class RoadSegmentationDataset(Dataset):
    """Dataset for DeepGlobe road segmentation."""
    
    def __init__(self, data_dir, img_size=512, normalize=True):
        self.data_dir = data_dir
        self.normalize = normalize
        
        # Get satellite images (ending with _sat.jpg)
        self.image_files = sorted([f for f in os.listdir(data_dir) if f.endswith('_sat.jpg')])
        
        # Transforms
        self.resize = transforms.Resize((img_size, img_size))
        self.to_tensor = transforms.ToTensor()
        self.normalize_transform = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    
    def __len__(self):
        return len(self.image_files)
    
    def __getitem__(self, idx):
        # Load image
        img_name = self.image_files[idx]
        img_path = os.path.join(self.data_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        
        # Load mask (*_sat.jpg -> *_mask.png)
        mask_name = img_name.replace('_sat.jpg', '_mask.png')
        mask_path = os.path.join(self.data_dir, mask_name)
        mask = Image.open(mask_path).convert('L')
        
        # Resize
        image = self.resize(image)
        mask = self.resize(mask)
        
        # Convert to tensor
        image = self.to_tensor(image)
        mask = self.to_tensor(mask)
        
        # Normalize image
        if self.normalize:
            image = self.normalize_transform(image)
        
        # Binary mask (0 = background, 1 = road)
        mask = (mask > 0.5).long().squeeze(0)
        
        return image, mask

print("✅ Dataset class created!")


## Step 5: Load DeepLabV3 Model


In [ ]:
import torch.nn as nn
from torchvision.models.segmentation import deeplabv3_resnet50

# Load pretrained DeepLabV3
model = deeplabv3_resnet50(weights='DEFAULT')

# Replace classifier head for 2 classes (road & background)
model.classifier[4] = nn.Conv2d(256, 2, kernel_size=1)
model.aux_classifier[4] = nn.Conv2d(256, 2, kernel_size=1)

# Move to GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"✅ Model loaded on {device}")


## Step 6: Train the Model with Checkpoints

Training saves checkpoints after each epoch. If interrupted, re-run to resume from last checkpoint.


In [ ]:
from torch.optim import Adam
from google.colab import files

# ============== CONFIGURATION ==============
# Save checkpoints locally in the Colab runtime (no Drive mount)
CHECKPOINT_PATH = "/content/checkpoints/checkpoint.pth"
# Auto-download after each epoch to avoid losing progress on runtime reset
DOWNLOAD_AFTER_EPOCH = True
NUM_EPOCHS = 10
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
# ===========================================

# Create dataset and dataloader
dataset = RoadSegmentationDataset(data_dir=train_dir, img_size=512)
print(f"Dataset size: {len(dataset)} images")

dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)

# ============== CHECKPOINT FUNCTIONS ==============
def save_checkpoint(epoch, model, optimizer, loss, path):
    """Save training checkpoint."""
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)
    print(f"💾 Checkpoint saved (Epoch {epoch + 1}) -> {path}")


def load_checkpoint(path, model, optimizer):
    """Load checkpoint and return starting epoch."""
    if os.path.exists(path):
        print(f"📂 Found checkpoint: {path}")
        checkpoint = torch.load(path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        print(f"✅ Resuming from epoch {start_epoch + 1}")
        return start_epoch
    return 0


def download_checkpoint(path):
    """Download the checkpoint to your machine to survive runtime resets."""
    if os.path.exists(path):
        print(f"⬇️ Downloading checkpoint: {path}")
        files.download(path)
    else:
        print(f"⚠️ No checkpoint found at {path} to download")


# Check for existing checkpoint
start_epoch = load_checkpoint(CHECKPOINT_PATH, model, optimizer)

# ============== TRAINING LOOP ==============
model.train()

print(f"\n🚀 Training from epoch {start_epoch + 1} to {NUM_EPOCHS}")
print("=" * 50)

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_loss = 0.0
    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    
    for batch_idx, (images, masks) in enumerate(progress_bar):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        output = model(images)['out']
        
        # Calculate loss
        loss = criterion(output, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Update progress bar
        epoch_loss += loss.item()
        avg_loss = epoch_loss / (batch_idx + 1)
        progress_bar.set_postfix(loss=f"{loss.item():.4f}", avg=f"{avg_loss:.4f}")
    
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} completed. Average Loss: {avg_loss:.4f}")
    
    # Save checkpoint after each epoch
    save_checkpoint(epoch, model, optimizer, avg_loss, CHECKPOINT_PATH)
    
    # Immediately download to avoid losing progress if runtime disconnects
    if DOWNLOAD_AFTER_EPOCH:
        download_checkpoint(CHECKPOINT_PATH)

print("\n✅ Training complete!")


## Step 7: Download checkpoints/models (no Drive)


In [ ]:
# Save final model locally, then download (no Drive mount needed)
import shutil

final_model_path = '/content/checkpoints/DeeplabsV3_road_final.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'epoch': NUM_EPOCHS,
    'model_type': 'deeplabv3_resnet50',
    'num_classes': 2
}, final_model_path)
print(f"✅ Final model saved to: {final_model_path}")

# Download final model to your machine
files.download(final_model_path)

# Download latest checkpoint (if exists)
if os.path.exists(CHECKPOINT_PATH):
    files.download(CHECKPOINT_PATH)

# Zip and download the entire checkpoints folder for convenience
if os.path.exists('/content/checkpoints'):
    archive_path = '/content/checkpoints.zip'
    shutil.make_archive('/content/checkpoints', 'zip', '/content/checkpoints')
    files.download(archive_path)
